In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os

In [3]:
load_dotenv(override=True)
AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1"
)
collection_name = "docs"
DEFAULT_MODEL = "Mistral-Large-3"
embedding_model = "qwen--qwen3-embedding-8b"
embedding_endpoint = "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1/chat/completions"

DB_NAME = "preprocessed_db"
KNOWLEDGE_BASE_PATH = Path("../knowledge_base")
AVERAGE_CHUNK_SIZE = 500


client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
)


In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )

    print(f"Loaded {len(documents)} documents")
    return documents


In [5]:
documents = fetch_documents()
print((documents[0]["text"]))


Loaded 76 documents
# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.

However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active contracts spanning all eight product lines. The company maintains 

In [6]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
import re

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)

headers = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]


c:\Users\ASUS\Desktop\scientific-paper-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class Result(BaseModel):
    page_content: str
    metadata: dict


In [8]:
def process_document(document):
    """
    Chunk documents differently depending on document type.

    Employees:
        Entire employee record = one chunk

    Contracts:
        Split by markdown headings

    Products:
        Split by markdown headings

    Company docs:
        Split by markdown headings

    Oversized sections:
        Recursively split.
    """

    doc_type = document["type"]

    # -----------------------------
    # Employee records
    # -----------------------------

    if doc_type.lower() == "employees":
        match = re.search(r"#\s+(.+)", document["text"])

        employee = match.group(1).strip() if match else "Unknown"

        return [
            Result(
                page_content=document["text"],
                metadata={
                    "source": document["source"],
                    "type": doc_type,
                    "employee": employee,
                },
            )
        ]

    # -----------------------------
    # Everything else
    # -----------------------------

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)

    sections = splitter.split_text(document["text"])

    results = []

    for section in sections:
        text = section.page_content.strip()

        metadata = {
            "source": document["source"],
            "type": doc_type,
            **section.metadata,
        }

        # -------------------------
        # Product name
        # -------------------------

        if doc_type.lower() == "products":
            match = re.search(r"#\s+(.+)", document["text"])

            if match:
                metadata["product"] = match.group(1).strip()

        # -------------------------
        # Contract customer
        # -------------------------

        elif doc_type.lower() == "contracts":
            match = re.search(
                r"Contract with (.+?) for",
                document["text"],
                re.IGNORECASE,
            )

            if match:
                metadata["customer"] = match.group(1).strip()

        # -------------------------
        # Small section
        # -------------------------

        if len(text) <= 700:
            results.append(
                Result(
                    page_content=text,
                    metadata=metadata,
                )
            )

        # -------------------------
        # Large section
        # -------------------------

        else:
            sub_chunks = recursive_splitter.create_documents(
                [text],
                metadatas=[metadata],
            )

            for chunk in sub_chunks:
                results.append(
                    Result(
                        page_content=chunk.page_content,
                        metadata=chunk.metadata,
                    )
                )

    return results


# Making Chunks


In [9]:
def create_chunks(documents):

    chunks = []

    for doc in tqdm(documents):
        chunks.extend(process_document(doc))

    return chunks


In [10]:
chunks = create_chunks(documents)


100%|██████████| 76/76 [00:00<00:00, 4100.74it/s]


In [12]:
print(chunks[0])

page_content='Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.  \nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': '../knowledge_base/company/about.md', 'type': 'company', 'h1': 'About Insurellm'}


In [13]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = client.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")


In [ ]:
create_embeddings(chunks)


[Result(page_content="Founding and Early Growth of Insurellm\n\nThis chunk covers the founding of Insurellm by Avery Lancaster in 2015 and its initial product, Markellm. It also details the company's rapid growth in its first five years, expanding its product portfolio and reaching 200 employees with 12 offices across the US by 2020.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.", metadata={'source': '../knowledge_base/company/about.md', 'type': 'company'}),
 Result(page_conte

### New approach: header-based + recursive chunking (no LLM calls, avoids Azure content filter)